# All reference models — one complete epoch

This Kaggle notebook trains every model registered in `REFERENCE_METHODS` for one **complete** epoch on UIEB and LSUI with seed 0, then evaluates the complete validation and test splits. It is a diagnostic comparison, not a replacement for the canonical 100-epoch benchmark. Results are written to a separate output root and are resumable.

## 1. Configuration

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/heniath/underwater-image-enhancement.git'
REPO_BRANCH = 'learnable-physics-extractor'
REPO_DIR = Path('/kaggle/working/underwater-image-enhancement')
UIEB_INPUT = Path('/kaggle/input/datasets/ohmahler91/uieb-dataset')
LSUI_INPUT = Path('/kaggle/input/datasets/ohmahler91/lsui-dataset/LSUI')
DATA_ROOT = Path('/kaggle/working/reference_data')
OUTPUT_ROOT = Path('/kaggle/working/reference_outputs_all_models_one_epoch')
TORCH_CACHE = Path('/kaggle/working/torch_cache')
PREVIOUS_OUTPUT_ROOT = None  # Optional attached output directory for resume.

DATASETS = ['UIEB', 'LSUI']
SEED = 0
RUN_TESTS = True
RUN_ONE_EPOCH = True
USE_RAM_CACHE = True


## 2. Repository and environment

In [ ]:
import importlib, os, shutil, subprocess, sys

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[dev,profile,visualization]'], check=True)
os.chdir(REPO_DIR)
repo_src = str(REPO_DIR / 'src')
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()

import torch, uwir
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('GPU:', torch.cuda.get_device_name(0))
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('uwir:', Path(uwir.__file__).resolve())
TORCH_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['UWIR_TORCH_HOME'] = str(TORCH_CACHE)


## 3. Datasets and optional resume

In [ ]:
def find_uieb(root):
    for candidate in [root] + [path.parent for path in root.rglob('raw-890')]:
        if (candidate / 'raw-890').is_dir() and (candidate / 'reference-890').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f'UIEB layout not found below {root}')

def link_dataset(name, target):
    link = DATA_ROOT / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise FileExistsError(f'Refusing to replace {link}')
    link.symlink_to(target, target_is_directory=True)

assert UIEB_INPUT.is_dir(), f'Missing UIEB input: {UIEB_INPUT}'
assert LSUI_INPUT.is_dir(), f'Missing LSUI input: {LSUI_INPUT}'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
link_dataset('UIEB', find_uieb(UIEB_INPUT))
link_dataset('LSUI', LSUI_INPUT.resolve())

if PREVIOUS_OUTPUT_ROOT is not None and not OUTPUT_ROOT.exists():
    previous = Path(PREVIOUS_OUTPUT_ROOT)
    assert previous.is_dir(), f'Missing previous output: {previous}'
    shutil.copytree(previous, OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('UIEB:', (DATA_ROOT / 'UIEB').resolve())
print('LSUI:', (DATA_ROOT / 'LSUI').resolve())
print('Output:', OUTPUT_ROOT)


## 4. Validate repository and discover all models

In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)

from uwir.reference_methods import REFERENCE_METHODS

METHODS = list(REFERENCE_METHODS)
assert METHODS, 'No reference methods are registered.'
print('Registered models:', METHODS)
print('Planned runs:', len(METHODS) * len(DATASETS), '(one seed, one epoch each)')


## 5. Build fixed UIEB/LSUI splits

In [ ]:
from uwir.datasets.lsui import build_lsui_datasets, discover_lsui
from uwir.datasets.uieb import build_uieb_datasets, discover_uieb
from uwir.training.runner import BenchmarkConfig, write_benchmark_metadata

CONFIG = BenchmarkConfig(epochs=1, model_seeds=(SEED,), cache_data='auto' if USE_RAM_CACHE else False)
assert CONFIG.epochs == 1
write_benchmark_metadata(OUTPUT_ROOT, CONFIG, REPO_DIR)

discovered = {}
if 'UIEB' in DATASETS:
    uieb_pairs = discover_uieb(DATA_ROOT / 'UIEB')
    assert len(uieb_pairs) == 890, f'Expected 890 UIEB pairs, got {len(uieb_pairs)}'
    discovered['UIEB'] = build_uieb_datasets(
        DATA_ROOT / 'UIEB', OUTPUT_ROOT / 'splits' / 'uieb_split_manifest.json', cache=CONFIG.cache_data
    )
if 'LSUI' in DATASETS:
    _, lsui_report = discover_lsui(DATA_ROOT / 'LSUI')
    assert lsui_report['matched_pairs'] == 4279, lsui_report
    assert not lsui_report['unmatched_inputs'] and not lsui_report['unmatched_gt'], lsui_report
    discovered['LSUI'] = build_lsui_datasets(
        DATA_ROOT / 'LSUI', OUTPUT_ROOT / 'splits' / 'lsui_split_manifest.json', cache=CONFIG.cache_data
    )
print({name: {split: len(data) for split, data in splits.items()} for name, splits in discovered.items()})


## 6. Train every registered model for one complete epoch

This intentionally uses `smoke=False`: every training batch and every validation/test image is processed. Completed model/dataset runs are skipped when resuming.

In [ ]:
from uwir.training.runner import run_reference_experiment

if RUN_ONE_EPOCH:
    for dataset_name in DATASETS:
        for method_name in METHODS:
            print(f'\n=== {dataset_name} | {method_name} | seed {SEED} | 1 epoch ===', flush=True)
            metrics = run_reference_experiment(
                dataset_name=dataset_name,
                method_name=method_name,
                datasets=discovered[dataset_name],
                output_root=OUTPUT_ROOT,
                repository_root=REPO_DIR,
                model_seed=SEED,
                config=CONFIG,
                device='cuda',
                smoke=False,
                resume=True,
            )
            print({key: value for key, value in metrics.items() if key != 'per_image'})
else:
    print('Dry run only. Set RUN_ONE_EPOCH=True to train.')


## 7. Results and completion check

In [ ]:
import pandas as pd
from uwir.training.runner import aggregate_results

results_path = OUTPUT_ROOT / 'per_run_results.csv'
expected = len(METHODS) * len(DATASETS)
completed = len(list(OUTPUT_ROOT.glob('*/*/seed_*/test_metrics.json')))
print('Completed:', completed, '/', expected)
if results_path.exists():
    aggregate_results(results_path, OUTPUT_ROOT / 'aggregate_results.csv')
    display(pd.read_csv(results_path).sort_values(['dataset', 'method', 'seed']))
    display(pd.read_csv(OUTPUT_ROOT / 'aggregate_results.csv'))
if RUN_ONE_EPOCH:
    assert completed == expected, f'Only {completed}/{expected} runs completed'
print('Save this directory as a Kaggle output or private Dataset:', OUTPUT_ROOT)
